# Session encoder and Two-Tower training

Issue: #24

## Question

Can a neural retriever, that puts a session and an item in the same vector space, find relevant items that co-visitation and popularity miss?

## Data used

Inputs: `data/sample/events.parquet` (#4), categories (#8, #9), and training queries built like the ranking queries of #10 from **all sessions of the training window (days 0-13)**, with every event cut at day 14 (a 400,000-query sample). No evaluation session (they start on day 14 or later) is used, and no event on or after day 14 is used. Evaluation queries: the 56,544 from #10 (shared rules from #17).

Outputs in `data/models/two_tower/`: `model.pt`, `config.json`, `item_embeddings.npy`, `catalog_aids.npy`. Code: `notebooks/twotower.py`.

**Catalog.** Items with at least 5 events before day 14 that have a category (the same information the co-visitation matrix of #7 uses). Items outside it cannot be retrieved; the coverage of the evaluation labels is printed.

**Model.** *Query tower*: the query category (one-hot for now, a classifier distribution in #36) becomes a weighted mix of category embeddings, joined with a session vector, the mean of the prefix item embeddings weighted by event type (click 1, cart 3, order 6) and recency (0.8 per step back); then an MLP to 64 dimensions. *Item tower*: item ID embedding + category embedding + popularity bucket embedding, then an MLP to 64 dimensions. Both outputs have unit length and the score is their dot product.

**Training.** Each example is a (session prefix, item that was later found) pair. Loss is a **sampled softmax with in-batch negatives**: the other items in the batch are the negatives. **logQ correction** subtracts `log(p_item)` from each item's logit, where `p_item` is how often the item is a positive.

## Why in-batch negatives are biased towards popular items

The negatives of an example are the positives of the other examples in the batch. So an item shows up as a negative in proportion to how often it is a positive, that is, to its popularity. A popular item is punished as a negative far more often than a rare one, only because it appears in more batches, and the model learns to push popular items down. Subtracting `log p_item` from the logit corrects for how likely the item was to be sampled as a negative (Yi et al., 2019), so the softmax behaves as if negatives were drawn uniformly. The notebook trains with and without the correction to see the effect.

In [ ]:
import json
import resource
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import torch

sys.path.insert(0, ".")
sys.path.insert(0, "notebooks")
from evalset import TOP, evaluate, prepare_eval, summarize
from metrics import recall_at_k
from queries import build_ranking_queries
from split import day_to_ms
from twotower import TwoTower, encode_queries, sampled_softmax_loss

DATA = Path("data") if Path("data").exists() else Path("../data")
OUT = DATA / "models" / "two_tower"
OUT.mkdir(parents=True, exist_ok=True)
D11, D14 = day_to_ms(11), day_to_ms(14)
SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.set_num_threads(4)

## Checks on a tiny problem first

1. The model can overfit 100 examples (loss close to zero).
2. The logQ correction lowers the logit of a popular item and changes the loss.
3. The same item twice in a batch is not treated as a negative.

In [ ]:
torch.manual_seed(0)
n_items, n_cats, n_pop, B = 300, 5, 4, 100
m = TwoTower(n_items, n_cats, n_pop, dim=16, cat_dim=8, pop_dim=4, hidden=32)
hist = torch.randint(0, n_items, (B, 6))
w = torch.ones(B, 6)
cat = torch.nn.functional.one_hot(torch.randint(0, n_cats, (B,)), n_cats).float()
items = torch.arange(B)                                  # 100 distinct positives
icat = torch.randint(0, n_cats, (n_items,))
ipop = torch.randint(0, n_pop, (n_items,))
opt = torch.optim.Adam(m.parameters(), lr=0.02)
first = None
for step in range(300):
    q = m.query(hist, w, cat)
    i = m.item(items, icat[items], ipop[items])
    loss = sampled_softmax_loss(q, i, items, temperature=0.1)
    if first is None:
        first = loss.item()
    opt.zero_grad(); loss.backward(); opt.step()
print(f"overfit test: loss {first:.3f} -> {loss.item():.4f}")
assert loss.item() < 0.05 * first

# logQ: item 0 is very popular (log_q high) -> its logit as a negative is lowered, so the loss changes
q = m.query(hist, w, cat).detach(); i = m.item(items, icat[items], ipop[items]).detach()
log_q = torch.full((n_items,), np.log(1 / n_items), dtype=torch.float32)
log_q[0] = np.log(0.5)
plain = sampled_softmax_loss(q, i, items, temperature=0.1)
corrected = sampled_softmax_loss(q, i, items, log_q=log_q, temperature=0.1)
assert plain.item() != corrected.item()
# For a row whose positive is not item 0, item 0 is a negative. Every logit is shifted by -log_q, and
# item 0 has a much larger log_q than the others, so relative to another item it is pushed DOWN.
logits = q @ i.T / 0.1
gap_before = logits[1, 0] - logits[1, 2]
gap_after = (logits[1, 0] - log_q[0]) - (logits[1, 2] - log_q[2])
assert gap_after < gap_before
print(f"logit gap (popular item 0 minus item 2): {gap_before.item():.3f} -> {gap_after.item():.3f}")
print("logQ changes the loss:", round(plain.item(), 4), "->", round(corrected.item(), 4))

# duplicates in a batch are masked: two rows with the same positive item must not use each other as negatives
dup_items = torch.tensor([7, 7, 8])
qd = m.query(hist[:3], w[:3], cat[:3]).detach(); idd = m.item(dup_items, icat[dup_items], ipop[dup_items]).detach()
loss_dup = sampled_softmax_loss(qd, idd, dup_items, temperature=0.1)
assert torch.isfinite(loss_dup)
print("toy checks pass")

## Catalog and training data

In [ ]:
events = pl.scan_parquet(DATA / "sample" / "events.parquet")
categories = pl.read_parquet(DATA / "categories" / "item_category.parquet")
names = json.loads((Path("artifacts") if Path("artifacts").exists() else Path("../artifacts")).joinpath("category_names.json").read_text())
n_cats = len(names["categories"])

before = events.filter(pl.col("ts") < D14)
stats = before.group_by("aid").agg(pl.len().alias("n"), (pl.col("type") == 0).sum().alias("clicks")).collect()
catalog = (
    categories.join(stats.filter(pl.col("n") >= 5), on="aid", how="inner").sort("aid")
)
catalog_aids = catalog["aid"].to_numpy()
n_items = len(catalog_aids)
index = {int(a): i for i, a in enumerate(catalog_aids)}
item_cat = torch.from_numpy(catalog["cluster"].to_numpy().astype(np.int64))
pop_bucket = np.minimum(15, np.floor(np.log2(1 + catalog["clicks"].to_numpy())).astype(np.int64))
item_pop = torch.from_numpy(pop_bucket)
n_pop = 16
print(f"catalog: {n_items:,} items with at least 5 events before day 14 and a category")

item_cluster = dict(zip(categories["aid"].to_list(), categories["cluster"].to_list(), strict=True))
train_events = events.filter(pl.col("ts") < D14).select("session", "aid", "ts", "type").sort("session", "ts").collect()
assert train_events["ts"].max() < D14
t0 = time.time()
train_q, tq_stats = build_ranking_queries(train_events, item_cluster, cap=10**9, seed=0)
del train_events
keep = np.sort(np.random.default_rng(SEED).choice(train_q.height, size=min(400_000, train_q.height), replace=False))
train_prepared, _ = prepare_eval(train_q[keep])
assert all(q["prefix_ts_last"] < D14 for q in train_prepared)
print(f"training queries: {tq_stats['usable']:,} usable from the whole training window, {len(train_prepared):,} kept after sampling and the relevance filter ({time.time() - t0:.0f} s)")

# examples: one (query, positive item) pair per relevant label item that is in the catalog
q_of, item_of = [], []
for qi, q in enumerate(train_prepared):
    rel = set().union(*q["labels"].values())
    for a in rel:
        j = index.get(int(a))
        if j is not None:
            q_of.append(qi); item_of.append(j)
q_of, item_of = np.array(q_of), np.array(item_of)
print(f"training examples: {len(q_of):,} from {len(np.unique(q_of)):,} queries")
hist_t, w_t, cat_t, cats_np = encode_queries(train_prepared, index, n_items, n_cats)

counts = np.bincount(item_of, minlength=n_items).astype(np.float64)
p_item = (counts + 1) / (counts.sum() + n_items)
log_q = torch.from_numpy(np.log(p_item)).float()

# hold out 5% of training queries to watch validation loss
rng = np.random.default_rng(SEED)
val_q = set(rng.choice(len(train_prepared), size=len(train_prepared) // 20, replace=False).tolist())
is_val = np.array([qi in val_q for qi in q_of])
tr_idx, va_idx = np.flatnonzero(~is_val), np.flatnonzero(is_val)
assert not (set(q_of[tr_idx]) & set(q_of[va_idx]))
print(f"train examples {len(tr_idx):,}, validation examples {len(va_idx):,}")

## Training loop

In [ ]:
def batch(ids):
    qi = torch.from_numpy(q_of[ids]); it = torch.from_numpy(item_of[ids])
    return qi, it


def train(logq, epochs=4, batch_size=2048, lr=2e-3, seed=SEED, label=""):
    torch.manual_seed(seed)
    model = TwoTower(n_items, n_cats, n_pop)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    rng = np.random.default_rng(seed)
    history = []
    best = (float("inf"), None, 0)
    start = time.time()
    for epoch in range(epochs):
        model.train()
        order = rng.permutation(tr_idx)
        losses = []
        for s in range(0, len(order) - batch_size + 1, batch_size):
            qi, it = batch(order[s : s + batch_size])
            q = model.query(hist_t[qi], w_t[qi], cat_t[qi])
            i = model.item(it, item_cat[it], item_pop[it])
            loss = sampled_softmax_loss(q, i, it, log_q=log_q if logq else None)
            opt.zero_grad(); loss.backward(); opt.step()
            losses.append(loss.item())
        model.eval()
        with torch.no_grad():
            v = []
            vo = rng.permutation(va_idx)
            for s in range(0, min(len(vo), 20_000) - batch_size + 1, batch_size):
                qi, it = batch(vo[s : s + batch_size])
                v.append(sampled_softmax_loss(model.query(hist_t[qi], w_t[qi], cat_t[qi]), model.item(it, item_cat[it], item_pop[it]), it, log_q=log_q if logq else None).item())
        history.append((epoch + 1, float(np.mean(losses)), float(np.mean(v))))
        if history[-1][2] < best[0]:
            best = (history[-1][2], {k: x.clone() for k, x in model.state_dict().items()}, epoch + 1)
        print(f"{label} epoch {epoch + 1}: train loss {history[-1][1]:.3f}, validation loss {history[-1][2]:.3f}")
    model.load_state_dict(best[1])          # keep the epoch with the lowest validation loss
    print(f"{label} best epoch {best[2]} (validation loss {best[0]:.3f}), trained in {time.time() - start:.0f} s")
    return model, history, time.time() - start


model, hist_logq, t_logq = train(True, label="with logQ")
model_plain, hist_plain, t_plain = train(False, label="no logQ  ")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
for h, name in ((hist_logq, "with logQ"), (hist_plain, "no logQ")):
    ax.plot([e for e, _, _ in h], [t for _, t, _ in h], "o-", label=f"{name} train")
    ax.plot([e for e, _, _ in h], [v for _, _, v in h], "s--", label=f"{name} validation")
ax.set_xlabel("epoch"); ax.set_ylabel("in-batch loss"); ax.legend(fontsize=7); ax.set_title("Training loss (lower is better)")
plt.show()
assert hist_logq[-1][1] < hist_logq[0][1], "training loss did not decrease"

## Retrieval on the evaluation queries

Inside the query category, every catalog item is scored by dot product (exact search). Prefix items are removed. Recall is measured at 20, 50, 100 and 200, and the top 100 are scored with the shared evaluator.

In [ ]:
eval_q = pl.read_parquet(DATA / "eval" / "ranking_queries.parquet")
prepared, prep_stats = prepare_eval(eval_q)
e_hist, e_w, e_cat, e_cats = encode_queries(prepared, index, n_items, n_cats)
by_cat = {c: np.flatnonzero(item_cat.numpy() == c) for c in range(n_cats)}

# how many relevant labels can the catalog reach at all?
rel_total = sum(len(set().union(*q["labels"].values())) for q in prepared)
rel_in = sum(sum(1 for a in set().union(*q["labels"].values()) if int(a) in index) for q in prepared)
print(f"relevant labels inside the catalog: {rel_in:,} of {rel_total:,} ({rel_in / rel_total:.1%})")


def retrieve(mdl, topn=200):
    mdl.eval()
    with torch.no_grad():
        item_vec = mdl.item(torch.arange(n_items), item_cat, item_pop)
        qvec = torch.cat([mdl.query(e_hist[s : s + 4096], e_w[s : s + 4096], e_cat[s : s + 4096]) for s in range(0, len(prepared), 4096)])
    out = {}
    for c in range(n_cats):
        rows = np.flatnonzero(e_cats == c)
        if not len(rows):
            continue
        members = by_cat[c]
        scores = qvec[rows] @ item_vec[members].T
        k = min(topn + 60, len(members))
        top = torch.topk(scores, k, dim=1).indices.numpy()
        for r, t in zip(rows, top, strict=True):
            prefix = prepared[r]["prefix"]
            lst = []
            for a in catalog_aids[members[t]]:
                if int(a) not in prefix:
                    lst.append(int(a))
                    if len(lst) == topn:
                        break
            out[prepared[r]["session"]] = lst
    return out, item_vec.numpy()


def score(mdl, name):
    lists, item_vec = retrieve(mdl)
    res = evaluate(prepared, lambda q: lists[q["session"]][:TOP])
    rec = {k: float(np.mean([recall_at_k(lists[q["session"]], set().union(*q["labels"].values()), k=k) for q in prepared])) for k in (20, 50, 100, 200)}
    s = summarize(res)
    print(f"{name}: NDCG@10 {s['ndcg10'][0]:.4f} [{s['ndcg10'][1]:.4f}, {s['ndcg10'][2]:.4f}]; recall " + ", ".join(f"@{k} {v:.4f}" for k, v in rec.items()))
    return res, rec, s, item_vec


res_logq, rec_logq, s_logq, item_vec = score(model, "Two-Tower with logQ")
res_plain, rec_plain, s_plain, _ = score(model_plain, "Two-Tower no logQ  ")
print(prep_stats)

## Compared with the baselines (recall of the exact in-category top lists)

In [ ]:
pop = pl.read_parquet(DATA / "results" / "popularity.parquet")
covis = pl.read_parquet(DATA / "results" / "covis.parquet")
table = pl.DataFrame({
    "method": ["popularity (#17)", "co-visitation + fill (#18)", "Two-Tower, no logQ", "Two-Tower, with logQ"],
    "ndcg10": [pop["ndcg10"].mean(), covis["ndcg10"].mean(), s_plain["ndcg10"][0], s_logq["ndcg10"][0]],
    "recall20": [pop["recall20"].mean(), covis["recall20"].mean(), rec_plain[20], rec_logq[20]],
    "recall100": [pop["recall100"].mean(), covis["recall100"].mean(), rec_plain[100], rec_logq[100]],
})
print(table)
print("Two-Tower (logQ) recall@100 vs popularity:", round(rec_logq[100], 4), "vs", round(pop["recall100"].mean(), 4))

In [ ]:
# save
torch.save(model.state_dict(), OUT / "model.pt")
np.save(OUT / "item_embeddings.npy", item_vec.astype(np.float32))
np.save(OUT / "catalog_aids.npy", catalog_aids)
cfg = {"dim": 64, "cat_dim": 32, "pop_dim": 16, "hidden": 128, "n_items": n_items, "n_cats": n_cats, "n_pop": n_pop,
       "temperature": 0.07, "epochs": 4, "batch_size": 2048, "lr": 2e-3, "seed": SEED, "logq": True,
       "catalog": "items with >= 5 events before day 14 and a category", "max_hist": 30,
       "train_seconds": {"logq": round(t_logq), "no_logq": round(t_plain)}, "embedding_shape": list(item_vec.shape)}
(OUT / "config.json").write_text(json.dumps(cfg, indent=2))
res_logq.write_parquet(DATA / "results" / "two_tower.parquet")
print("saved; item embeddings", item_vec.shape, "float32 =", round(item_vec.nbytes / 1e6), "MB; training time", round(t_logq), "s")
print("peak memory GB:", round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e9, 2))

## Result

Numbers are in the printouts above. On the 56,544 evaluation queries (exact in-category search, prefix items removed):

| Method | NDCG@10 | recall@20 | recall@50 | recall@100 | recall@200 |
|---|---|---|---|---|---|
| Two-Tower **with logQ** | 0.0459 | 0.1206 | 0.1973 | 0.2745 | 0.3659 |
| Two-Tower without logQ | 0.0181 | 0.0502 | 0.0908 | 0.1353 | 0.1913 |
| popularity (#17) | 0.0419 | 0.1172 | 0.1968 | 0.2773 | (0.3777 in #21) |
| co-visitation + fill (#18) | 0.2144 | 0.3885 | 0.4880 | 0.5553 | |

- **The Two-Tower is about as good as popularity, not better.** Its recall@100 (0.2745) is very slightly *below* popularity (0.2773); recall@20 and NDCG@10 are very slightly above. It is far below co-visitation. The issue asked for recall@100 above popularity; **that is not met**, and the reasons follow.
- **The logQ correction matters a lot.** Without it, recall@100 halves (0.135). This is the popularity bias from the section above made visible: in-batch negatives punish popular items, so an uncorrected model under-ranks exactly the items that are often relevant.
- **It overfits after one epoch.** The training loss keeps falling, the validation loss is lowest after epoch 1 and then rises (3.37, 3.40, 3.62, 3.93 with logQ), so the kept model is the epoch-1 one, trained on about 0.5 million examples. The item ID embeddings (395,000 x 64 = 25 million parameters) can memorise pairs, and most items have very few positives. A first attempt with only days 11-13 as training data (153,000 examples, 5 epochs) was worse still (recall@100 0.183), so more data helped.
- **Why it does not beat co-visitation.** The session is summarised by an average of item vectors, which loses the specific pairs (which item followed which) that co-visitation counts exactly. For a session with a few events, the exact pair counts are a stronger signal than a 64-dimensional summary learned from half a million examples.
- **Not tried:** lower learning rate or weight decay, dropout on the ID embedding, a larger batch for more negatives, more epochs with regularisation, a transformer session encoder. Hard negatives are the next issue (#25). The Two-Tower can still add candidates that co-visitation and popularity miss, which is measured when it joins the pool in #26.

## What I learned

To be written by the owner of the project.